# Erion Ember x Qwen 3.5 Playground

<a href="https://colab.research.google.com/github/EricNguyen1206/erion-ember/blob/main/docs/templates/erion_ember_qwen35_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook allows you to test the semantic caching capabilities of Erion Ember using the Qwen3.5-0.8B model.

In [ ]:
# @title (Optional) Mount Google Drive
# Use this if you want to save your progress or logs to your Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Install Dependencies & Download Server
!pip install -q transformers accelerate requests ipywidgets

# Replace with your actual release URL if needed, or use the latest from main branch binary
!wget -q https://github.com/EricNguyen1206/erion-ember/releases/latest/download/erion-ember -O erion-ember || \
 (echo "Release binary not found, attempting to build from source..." && \
  add-apt-repository ppa:longsleep/golang-backports -y && apt update && apt install golang-go -y && \
  git clone https://github.com/EricNguyen1206/erion-ember.git && cd erion-ember && go build -o ../erion-ember ./cmd/server/)

!chmod +x erion-ember

import subprocess
import time
import requests

# Start server in background
subprocess.Popen(["./erion-ember"], stdout=open("server.log", "w"), stderr=subprocess.STDOUT)

# Wait for server to be ready
for _ in range(15):
    try:
        resp = requests.get("http://localhost:8080/health")
        if resp.status_code == 200:
            print("✅ Erion Ember server is ready!")
            break
    except:
        time.sleep(1)
else:
    print("❌ Server failed to start. Check server.log")

In [ ]:
# @title Load Qwen3.5-0.8B
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen2.5-0.8B" # Note: Using Qwen2.5 until 3.5-0.8B is officially on HF
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # Using a shorter max_new_tokens for the demo
    outputs = model.generate(**inputs, max_new_tokens=64, do_sample=True, temperature=0.7)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"✅ Model {model_id} loaded successfully!")

In [ ]:
# @title Interactive Playground
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt1_input = widgets.Textarea(placeholder='Enter initial prompt to cache...', description='Prompt 1:', layout={'width': '80%','height': '100px'})
prompt2_input = widgets.Textarea(placeholder='Enter similar prompt to test cache...', description='Prompt 2:', layout={'width': '80%','height': '100px'})
threshold_slider = widgets.FloatSlider(value=0.85, min=0.0, max=1.0, step=0.01, description='Similarity:')
cache_btn = widgets.Button(description="Cache Prompt 1", button_style='info')
test_btn = widgets.Button(description="Test Prompt 2", button_style='success')
output_area = widgets.Output()

def on_cache_click(b):
    with output_area:
        clear_output()
        prompt = prompt1_input.value
        if not prompt: return print("Please enter a prompt.")
        
        print(f"Generating response with Qwen...")
        response = generate_response(prompt)
        
        print(f"Caching in Erion Ember...")
        requests.post("http://localhost:8080/v1/cache/set", json={"prompt": prompt, "response": response})
        print(f"✅ Cached successfully!")
        print(f"Tokens (Prompt+Response): {len(tokenizer.encode(prompt + response))}")
        print(f"Cached Response: {response[:200]}...")

def on_test_click(b):
    with output_area:
        clear_output()
        prompt = prompt2_input.value
        if not prompt: return print("Please enter a prompt.")
        
        start = time.time()
        try:
            resp = requests.post("http://localhost:8080/v1/cache/get", json={
                "prompt": prompt, 
                "similarity_threshold": threshold_slider.value
            }).json()
        except Exception as e:
            return print(f"Error communicating with server: {e}")

        latency = (time.time() - start) * 1000
        
        if resp.get("hit"):
            tokens = len(tokenizer.encode(resp['response']))
            print(f"🎯 CACHE HIT!")
            print(f"Similarity: {resp['similarity']:.4f}")
            print(f"Latency: {latency:.2f}ms")
            print(f"Tokens Saved: {tokens}")
            print(f"Cached Response: {resp['response']}")
        else:
            print(f"❌ CACHE MISS")
            print(f"Latency: {latency:.2f}ms")
            print("Generating new response...")
            new_resp = generate_response(prompt)
            print(f"Result: {new_resp}")

cache_btn.on_click(on_cache_click)
test_btn.on_click(on_test_click)

display(prompt1_input, cache_btn, widgets.HTML("<hr>"), prompt2_input, threshold_slider, test_btn, output_area)